# LLM 09. Generate Reporting Topic Groups

카테고리/긍부정 그룹별 세부 topic 10-15개를 GPT-5-5로 3-5개 상위 topic_group으로 묶습니다.

- `기타`, `미분류`, `전반적 긍정/부정`, `LLM_FALLBACK_REQUIRED`는 모두 `기타` 그룹으로 통합합니다.
- 세부 topic은 topic명과 description을 함께 보고 의미가 가까운 것끼리 묶습니다.
- 이 노트북에서는 `topic_group` 매핑 테이블만 생성/검증합니다.
- 원본 row에 topic_group까지 붙이는 Tableau 최종 산출물은 15번 노트북에서 생성합니다.


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import taxonomy.topic_group_generator as topic_group_generator

importlib.reload(config_loader)
importlib.reload(topic_group_generator)

from common.config_loader import load_config, get_output_table, get_reference_table
from taxonomy.topic_group_generator import (
    generate_and_save_topic_groups,
    load_latest_topic_group_df,
)

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

print("topic_pool =", get_output_table(config, "topic_pool"))
print("topic_group =", get_output_table(config, "topic_group"))


In [ ]:
# 실행 옵션
# 기본값은 전체 그룹 실행입니다. 이미 생성된 그룹은 SKIP_EXISTING=True로 재호출하지 않습니다.
LIMIT_GROUPS = None
MODEL_KEY = "gpt_55"
SKIP_EXISTING = True

result = generate_and_save_topic_groups(
    spark,
    config,
    limit_groups=LIMIT_GROUPS,
    model_key=MODEL_KEY,
    skip_existing=SKIP_EXISTING,
    write_mode="replace_groups",
)

result


In [ ]:
# Topic -> Topic Group 매핑 확인
topic_group_table = get_output_table(config, "topic_group")
category_mapping_table = get_reference_table(config, "category_mapping_table")

topic_group_df = load_latest_topic_group_df(spark, config)
mapping_df = spark.table(category_mapping_table).select(
    "cate_1_depth",
    "cate_2_depth",
    "cate_1_depth_kor",
    "cate_2_depth_kor",
).dropDuplicates(["cate_1_depth", "cate_2_depth"])

display(
    topic_group_df.alias("g")
    .join(
        mapping_df.alias("m"),
        on=["cate_1_depth", "cate_2_depth"],
        how="left",
    )
    .select(
        "g.cate_1_depth",
        "m.cate_1_depth_kor",
        "g.cate_2_depth",
        "m.cate_2_depth_kor",
        "g.sc_measurement",
        "g.topic_group_order",
        "g.topic_group",
        "g.topic",
        "g.topic_description",
        "g.grouping_reason",
        "g.is_special_group",
    )
    .orderBy("g.cate_1_depth", "g.cate_2_depth", "g.sc_measurement", "g.topic_group_order", "g.topic_order")
)


In [ ]:
# 카테고리/긍부정/topic_group/topic별 최종 detail 메모 수 전체 확인
final_detail_table = get_output_table(config, "classification_detail_final")
topic_group_table = get_output_table(config, "topic_group")
category_mapping_table = get_reference_table(config, "category_mapping_table")

final_detail_df = spark.table(final_detail_table).where(F.col("prompt_version") == config["version"]["prompt_version"])
topic_group_df = load_latest_topic_group_df(spark, config)
mapping_df = spark.table(category_mapping_table).select(
    "cate_1_depth",
    "cate_2_depth",
    "cate_1_depth_kor",
    "cate_2_depth_kor",
).dropDuplicates(["cate_1_depth", "cate_2_depth"])

topic_count_df = (
    final_detail_df.alias("d")
    .join(
        topic_group_df.alias("g"),
        on=[
            F.col("d.cate_1_depth") == F.col("g.cate_1_depth"),
            F.col("d.cate_2_depth") == F.col("g.cate_2_depth"),
            F.col("d.sc_measurement") == F.col("g.sc_measurement"),
            F.col("d.pred_topic") == F.col("g.topic"),
        ],
        how="left",
    )
    .join(
        mapping_df.alias("m"),
        on=[
            F.col("d.cate_1_depth") == F.col("m.cate_1_depth"),
            F.col("d.cate_2_depth") == F.col("m.cate_2_depth"),
        ],
        how="left",
    )
    .groupBy(
        F.col("d.cate_1_depth").alias("cate_1_depth"),
        F.col("m.cate_1_depth_kor").alias("cate_1_depth_kor"),
        F.col("d.cate_2_depth").alias("cate_2_depth"),
        F.col("m.cate_2_depth_kor").alias("cate_2_depth_kor"),
        F.col("d.sc_measurement").alias("sc_measurement"),
        F.coalesce(F.col("g.topic_group"), F.lit("기타")).alias("topic_group"),
        F.coalesce(F.col("g.topic_group_order"), F.lit(999)).alias("topic_group_order"),
        F.col("d.pred_topic_type").alias("pred_topic_type"),
        F.col("d.pred_topic").alias("pred_topic"),
    )
    .agg(
        F.count("*").alias("memo_cnt"),
        F.countDistinct("d.memo_id").alias("distinct_memo_id_cnt"),
        F.avg("d.confidence_score").alias("avg_confidence"),
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic_group_order", F.desc("memo_cnt"))
)

display(topic_count_df)
